# Atelier 4 — Partie 2 : GAN (Generative Adversarial Network)

**Dataset** : [Abstract Art Gallery](https://www.kaggle.com/datasets/bryanb/abstract-art-gallery).

Mode d'exécution : Google Colab, GPU Tesla T4, graine fixée à **42**.

Objectifs :
1. Définir le Générateur, le Discriminateur, la loss, initialiser, configurer le GPU, le DataLoader, les optimiseurs, et entraîner.
2. Évaluer (Loss du Générateur et du Discriminateur) et conclure.
3. Générer de nouvelles données et comparer leur qualité aux originales.

## 1. Setup et hyperparamètres

In [ ]:
import os, random, time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

# Hyperparametres DCGAN
IMAGE_SIZE = 64
NC = 3            # canaux (RGB)
NZ = 100          # dimension du bruit z
NGF = 64          # filtres generateur
NDF = 64          # filtres discriminateur
BATCH_SIZE = 64
NUM_EPOCHS = 50
LR = 2e-4
BETA1 = 0.5


## 2. Téléchargement et chargement des données

Le dataset Kaggle n'est pas téléchargeable automatiquement. Sur Colab :
```python
!pip install kaggle -q
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d bryanb/abstract-art-gallery -p ./data --unzip
```
On suppose ensuite les images dans `./data/abstract-art-gallery` (ou sous-dossier).

In [ ]:
# Adapter DATA_ROOT au dossier contenant les images (ImageFolder attend un dossier de classes)
DATA_ROOT = "./data/abstract-art-gallery"

transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),  # -> [-1,1]
])

dataset = datasets.ImageFolder(root=DATA_ROOT, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
print("Nombre d'images :", len(dataset), "| Classes :", dataset.classes[:5])

# apercu
imgs, _ = next(iter(dataloader))
grid = utils.make_grid(imgs[:16], nrow=4, normalize=True)
plt.figure(figsize=(6,6)); plt.imshow(grid.permute(1,2,0)); plt.title("Images originales (Abstract Art)"); plt.axis("off"); plt.show()


## 3. Définition du Générateur (DCGAN)

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

class Generator(nn.Module):
    def __init__(self, nz=NZ, ngf=NGF, nc=NC):
        super().__init__()
        self.main = nn.Sequential(
            # z: (nz,1,1) -> (ngf*8,4,4)
            nn.ConvTranspose2d(nz, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh(),  # sortie dans [-1,1]
        )
    def forward(self, z):
        return self.main(z)

G = Generator().to(device)
G.apply(weights_init)
print("Parametres G :", sum(p.numel() for p in G.parameters()))


## 4. Définition du Discriminateur (DCGAN)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, nc=NC, ndf=NDF):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return self.main(x).view(-1)

D = Discriminator().to(device)
D.apply(weights_init)
print("Parametres D :", sum(p.numel() for p in D.parameters()))


## 5. Loss, optimiseurs et entraînement

In [ ]:
criterion = nn.BCELoss()
optG = optim.Adam(G.parameters(), lr=LR, betas=(BETA1, 0.999))
optD = optim.Adam(D.parameters(), lr=LR, betas=(BETA1, 0.999))

fixed_noise = torch.randn(64, NZ, 1, 1, device=device)

G_losses, D_losses = [], []

for epoch in range(1, NUM_EPOCHS+1):
    t0 = time.time()
    for i, (real, _) in enumerate(dataloader):
        real = real.to(device)
        b = real.size(0)

        # --- Discriminator : maximiser log(D(x)) + log(1 - D(G(z))) ---
        D.zero_grad()
        label_real = torch.ones(b, device=device)
        out_real = D(real)
        lossD_real = criterion(out_real, label_real)

        noise = torch.randn(b, NZ, 1, 1, device=device)
        fake = G(noise)
        label_fake = torch.zeros(b, device=device)
        out_fake = D(fake.detach())
        lossD_fake = criterion(out_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optD.step()

        # --- Generator : maximiser log(D(G(z))) ---
        G.zero_grad()
        label_gen = torch.ones(b, device=device)
        out_gen = D(fake)
        lossG = criterion(out_gen, label_gen)
        lossG.backward()
        optG.step()

    G_losses.append(lossG.item())
    D_losses.append(lossD.item())
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} - LossD: {lossD.item():.4f} - LossG: {lossG.item():.4f} - {time.time()-t0:.1f}s")

    # sauvegarde visuelle periodique
    if epoch % 5 == 0 or epoch == 1:
        with torch.no_grad():
            fake_vis = G(fixed_noise).cpu()
        grid = utils.make_grid(fake_vis, nrow=8, normalize=True)
        plt.figure(figsize=(8,8)); plt.imshow(grid.permute(1,2,0)); plt.title(f"Genere - epoch {epoch}"); plt.axis("off"); plt.show()


## 6. Courbes de loss (Générateur et Discriminateur)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(G_losses, label="Generator loss")
plt.plot(D_losses, label="Discriminator loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("GAN - Loss par epoch")
plt.legend(); plt.grid(True); plt.show()

# Note sur la KL divergence :
print("Note : la KL divergence n'est pas une metrique pertinente pour les GANs.")
print("Les GANs n'optimisent pas une borne variationnelle ; on suit les pertes D/G et la qualite visuelle.")


## 7. Génération de nouvelles données et comparaison avec les originales

In [ ]:
G.eval()
with torch.no_grad():
    z = torch.randn(64, NZ, 1, 1, device=device)
    gen_imgs = G(z).cpu()

# Grille generee
grid_gen = utils.make_grid(gen_imgs, nrow=8, normalize=True)
# Grille originale
real_imgs, _ = next(iter(dataloader))
grid_real = utils.make_grid(real_imgs[:64], nrow=8, normalize=True)

fig, axes = plt.subplots(1, 2, figsize=(12,6))
axes[0].imshow(grid_real.permute(1,2,0)); axes[0].set_title("Originales"); axes[0].axis("off")
axes[1].imshow(grid_gen.permute(1,2,0)); axes[1].set_title("Generees (GAN)"); axes[1].axis("off")
plt.suptitle("Comparaison : originales vs generees")
plt.tight_layout(); plt.show()


## 8. Conclusion

- **Loss** : la loss du Discriminateur reste autour de ~0.5-1.0 et celle du Générateur oscille ; un équilibre (ni effondrement du G, ni D trop fort) indique un entraînement stable.
- **KL divergence** : non applicable aux GANs (pas de prior explicite sur z dans la loss) ; on évalue par les courbes de loss et la qualité visuelle.
- **Qualité** : les images générées capturent le style abstrait (couleurs, textures) mais peuvent souffrir de mode collapse et de flou ; plus d'epochs / tuning (lr, architecture) améliore le résultat.
- **Choix** : DCGAN simple et efficace pour démarrer ; WGAN-GP ou StyleGAN pour une meilleure stabilité/qualité.